In [10]:
import sys
from SciExpeM_API.SciExpeM import SciExpeM
from SciExpeM_API.Models import *

# Fill in your credentials (or use token=...)
my_sciexpem = SciExpeM(username=USERNAME, password=PASSWORD)

Attention. SciExpeM is a singleton.


### Common properties

- name
- units
- value
- source_type

In [72]:
P = 12 #atm
phi = 0.25 #unitless
# c3 = CommonProperty(name='temperature', units='K', value='1098', source_type='reported')
c1 = CommonProperty(name='pressure', units='atm', value=str(P), source_type='reported')
#c2 = CommonProperty(name='residence time', units='s', value='0.004', source_type='reported')
# if parametric analysis, report only the common feature, see other examples
commonprop = [c1,]

### Initial Species

- name
- units
- value
- source_type
- configuration

In [73]:
# Initial composition of the mixture (single composition for an IDT experiment).
# PREMIXED is the default configuration unless it's a counterflow (CF) flame.
species = ['C7H8', 'O2', 'N2']
composition = ['0.00580', '0.20887', '0.78533']
comp_unit = 'mole fraction'
srctype = 'reported'
config = 'premixed'
################# do not edit
inspecies = []
species_objects = {}
for i, s in enumerate(species):
    si = my_sciexpem.filterDatabase(model_name='Species', preferredKey=s)[0]  # NB 'Species'
    species_objects[s] = si
    ii = InitialSpecies(name=species[i],       # NB class is InitialSpecies
                        species=si,            # NB kwarg is `species`
                        units=comp_unit,
                        value=composition[i],
                        source_type=srctype,
                        configuration=config)
    inspecies.append(ii)

### Data columns

- name
- label
- units
- data
- dg_id 
- dg_label
- source_type


In [74]:
T = [1152,1169,1171,1191,1220,1230,1231,1263,1268,1291,1385,1400,]
IDT_MUS = [966,600,687,639,465,398,475,342,298,233,151,94,]

# --- (Optional) uncertainty on x and/or y ---
# Create a DataColumn named 'uncertainty' and point the parent column at it via
# uncertainty_reference. IMPORTANT: uncertainty_kind ('absolute'/'relative') and
# uncertainty_bound ('plusminus'/'plus'/'minus') go on the UNCERTAINTY column;
# it must share the same dg_id, units and length as the column it refers to.
unc_T = DataColumn(name='uncertainty', units='K',
                   dg_id='dg1', dg_label='experimental_data',
                   data=[10] * len(T), source_type='reported',
                   uncertainty_kind='absolute', uncertainty_bound='plusminus')

unc_idt = DataColumn(name='uncertainty', units='us',
                     dg_id='dg1', dg_label='experimental_data',
                     data=[0.1] * len(IDT_MUS), source_type='reported',
                     uncertainty_kind='relative', uncertainty_bound='plusminus')

# x column (temperature) with its uncertainty
d1 = DataColumn(name='temperature', units='K',
                dg_id='dg1', dg_label='experimental_data',
                data=T, source_type='reported',
                uncertainty_reference=unc_T)     # <-- x uncertainty (omit to skip)

# y column (ignition delay) with its uncertainty
d2 = DataColumn(name='ignition delay', units='us',
                dg_id='dg1', dg_label='experimental_data',
                data=IDT_MUS, source_type='reported',
                uncertainty_reference=unc_idt)    # <-- y uncertainty (omit to skip)

datacols = [d1, d2]

### FilePaper
 - description*
 - reference_doi*
 - author*
 - title*
 - year*
 - volume
 - page
 - journal

In [36]:
# QUESTO PIù COMODO DA FARE DIRETTAMENTE SU SCIEXPEM E POI FAI COPIA INCOLLA QUI
file_paper = FilePaper(reference_doi="10.1016/J.PROCI.2008.05.004", 
                       author="Shen, Hsi-Ping S.; Vanderover, Jeremy; Oehlschlaeger, Matthew A.",                       
                       title="Probing The Antagonistic Effect Of Toluene As A Component In Surrogate Fuel Models At Low Temperatures And High Pressures. A Case Study Of Toluene/Dimethyl Ether Mixtures",
                       year="2009",
                       description="Shen, Hsi-Ping S.; Vanderover, Jeremy; Oehlschlaeger, Matthew A. - Proceedings Of The Combustion Institute, 2009, (32), 165-172",
                       )

### OpenSMOKE input file if available

In [75]:
input_path = r'D:\POLI\OPENSMOKE\TOLUENE\IDT\Shen_2009\12 atm\025\input.dic'
with open(input_path) as f:
    inputstr = f.read()
# NB è MEGLIO METTERE TUTTO SULLA STESSA RIGA!! QUINDI TIPO INPUT DEVE ESSERE COSì

### Assembly the experiment

In [76]:
e = Experiment(
    reactor='shock tube',
    experiment_type='ignition delay measurement',
    ignition_type='p-d/dt max',
    reactor_modes=['reflected shock'],
    file_paper=file_paper,
    data_columns=datacols,
    initial_species=inspecies,
    common_properties=commonprop,
    os_input_file=inputstr,
    t_inf=min(T), t_sup=max(T),          # characteristics (T range of the sweep)
    p_inf=P, p_sup=P,
    phi_inf=phi, phi_sup=phi,
    fuels_object=[species_objects['C7H8'].id],   # list of Species IDs (NOT the old `fuels` string field)
)

In [ ]:
e.serialize()
# QUESTO TI FA VEDERE IL JSON DELLE COSE CHE STAI CARICANDO 
# IN MODO DA CONTROLLARE CHE SIA TUTTO A POSTO

### Send Experiment

In [77]:
my_sciexpem.insertElement(e, verbose=True)
# QUI DOVRESTI RIUSCIRE A FARLO SMOOTHLY

Experiment element inserted successfully.
